In [ ]:
"""
Shannon Diversity Index calculation
------------------------------------
Formula (Shannon, 1948):
    H = - sum_i ( p_i * ln(p_i) )

where p_i is the relative abundance (proportion) of taxon i in a sample,
and the sum runs over all taxa present in that sample.

The input file "Rel_ab_codif.csv" already contains relative abundances
(p_i values) for each taxon ("seq") in each sample column, so no
additional normalization is required. By convention, terms where p_i = 0
contribute 0 to the sum (since lim_{p->0} p*ln(p) = 0).
"""

import pandas as pd
import numpy as np

# ---- Load data -------------------------------------------------------
df = pd.read_csv("/mnt/project/Rel_ab_codif.csv")

# Identify the sample columns (everything after the taxonomy/annotation columns)
non_sample_cols = ["seq", "Kingdom", "Phylum", "Class", "Order",
                    "Family", "Genus", "Species", "Combined", "Combined.1"]
# Note: the file has two columns literally named "Combined"; pandas
# auto-renames the duplicate to "Combined.1", so both are excluded above.
sample_cols = [c for c in df.columns if c not in non_sample_cols]

print(f"Detected {len(sample_cols)} sample columns: {sample_cols}\n")

# ---- Sanity check: do relative abundances sum to 1 per sample? -------
sums = df[sample_cols].sum(axis=0)
print("Column sums (should be ~1.0 if these are proper relative abundances):")
print(sums.round(6).to_string())
print()

# ---- Shannon diversity index calculation ------------------------------
def shannon_index(p):
    """Compute H = -sum(p_i * ln(p_i)), treating p_i = 0 as contributing 0.
    p_i must be a true proportion (sums to 1) -- the raw column is
    normalized by its own sum first in case values are given as
    percentages (summing to 100) rather than proportions (summing to 1)."""
    p = p[p > 0]  # ignore zero-abundance taxa (0 * ln(0) -> 0 by convention)
    p = p / p.sum()  # normalize to true proportions (handles % vs fraction)
    return -np.sum(p * np.log(p))

results = {}
for col in sample_cols:
    results[col] = shannon_index(df[col].values)

shannon_df = pd.Series(results, name="Shannon_H").sort_index()

# ---- Merge with day-of-operation metadata for readability -------------
day_map = pd.read_csv("/mnt/project/Sample_ID_to_day_of_operation.csv")
# Note: sample "46152" in Rel_ab_codif.csv corresponds to "10-May" in the
# day-of-operation file (an Excel date-serialization artifact of the same
# original sample ID), which maps to Day of Operation = 0.
day_map["Sample ID"] = day_map["Sample ID"].replace({"10-May": "46152"})
day_map = day_map.set_index("Sample ID")["Day of Operation"]

results_table = pd.DataFrame({
    "Shannon_H": shannon_df,
    "Day_of_Operation": day_map.reindex(shannon_df.index)
}).sort_values("Day_of_Operation")

print("Shannon Diversity Index (H) per sample, ordered by day of operation:\n")
print(results_table.to_string())

# ---- Save results ------------------------------------------------------
results_table.to_csv("/mnt/user-data/outputs/shannon_diversity_results.csv")
print("\nSaved results to shannon_diversity_results.csv")